In [5]:
import json
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd

# 域列表：domain ∈ [clapnq, fiqa, cloud, govt]
domains = ["clapnq", "fiqa", "cloud", "govt"]

# 字段名配置（如有不同请在此修改）
orig_field = "text"   # 原始 query 字段名（*_rewrite.jsonl 里的字段）
rew_field = "text"  # 改写后 query 字段名（*_rewrite_gpt.jsonl 里的字段）

# 简单分词（按空格）
def tokenize(text: str):
    if not isinstance(text, str):
        return []
    return text.strip().split()

# 代词列表
PRONOUNS = {
    "he", "she", "it", "they",
    "this", "that", "those", "these",
    "him", "her", "them",
}


def count_pronouns(tokens):
    return sum(1 for t in tokens if t.lower() in PRONOUNS)


def pronoun_counter(tokens):
    c = Counter()
    for t in tokens:
        t_low = t.lower()
        if t_low in PRONOUNS:
            c[t_low] += 1
    return c


def analyze_domain(domain: str):
    """对单个 domain 的 original vs rewrite_gpt 做长度 & 代词比例统计。

    使用：
    - cleaned_dataset/{domain}/{domain}_rewrite.jsonl        作为 original
    - cleaned_dataset/{domain}/{domain}_rewrite_gpt.jsonl     作为 rewritten
    通过 _id / task_id 对齐。
    """
    orig_path = Path(f"cleaned_dataset/{domain}/{domain}_rewrite.jsonl")
    rew_path = Path(f"cleaned_dataset/{domain}/{domain}_rewrite_gpt.jsonl")

    if not orig_path.exists():
        print(f"[WARN] 跳过 {domain}: original 文件不存在 -> {orig_path}")
        return
    if not rew_path.exists():
        print(f"[WARN] 跳过 {domain}: rewrite_gpt 文件不存在 -> {rew_path}")
        return

    # 读取 original
    orig_records = []
    with orig_path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            orig_records.append(json.loads(line))

    # 读取 rewrite_gpt
    rew_records = []
    with rew_path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            rew_records.append(json.loads(line))

    if not orig_records or not rew_records:
        print(f"[WARN] {domain}: original 或 rewrite_gpt 文件为空")
        return

    df_orig = pd.DataFrame(orig_records)
    df_rew = pd.DataFrame(rew_records)

    # 统一 id 字段名
    if "task_id" in df_orig.columns:
        df_orig = df_orig.rename(columns={"task_id": "_id"})
    if "task_id" in df_rew.columns:
        df_rew = df_rew.rename(columns={"task_id": "_id"})

    if "_id" not in df_orig.columns or "_id" not in df_rew.columns:
        print(f"[WARN] {domain}: original 或 rewrite 缺少 _id/task_id 字段")
        print("  original 列:", list(df_orig.columns))
        print("  rewrite  列:", list(df_rew.columns))
        return

    # inner join，仅保留两边都存在的样本
    df = df_orig[["_id", orig_field]].merge(
        df_rew[["_id", rew_field]], on="_id", how="inner", suffixes=("_orig", "_rew")
    )

    if df.empty:
        print(f"[WARN] {domain}: original 与 rewrite_gpt 无交集样本")
        return

    # 为原始 / 改写分别计算长度和代词比例
    orig_tokens = df[f"{orig_field}_orig"].map(tokenize)
    rew_tokens = df[f"{rew_field}_rew"].map(tokenize)

    df["orig_len"] = orig_tokens.map(len)
    df["rew_len"] = rew_tokens.map(len)

    df["orig_pronoun_cnt"] = orig_tokens.map(count_pronouns)
    df["rew_pronoun_cnt"] = rew_tokens.map(count_pronouns)

    # 代词比例 = pronoun_count / total_tokens，长度为 0 时记为 NaN
    df["orig_pronoun_ratio"] = df["orig_pronoun_cnt"] / df["orig_len"].replace(0, np.nan)
    df["rew_pronoun_ratio"] = df["rew_pronoun_cnt"] / df["rew_len"].replace(0, np.nan)

    # corpus 级别的 pronoun 统计
    orig_pron_counter = Counter()
    rew_pron_counter = Counter()
    for toks in orig_tokens:
        orig_pron_counter.update(pronoun_counter(toks))
    for toks in rew_tokens:
        rew_pron_counter.update(pronoun_counter(toks))

    # 长度统计：均值 & 中位数
    length_stats = pd.DataFrame({
        "mean_len": {
            "original": df["orig_len"].mean(),
            "rewritten": df["rew_len"].mean(),
        },
        "median_len": {
            "original": df["orig_len"].median(),
            "rewritten": df["rew_len"].median(),
        },
    })

    # 代词比例统计（按样本平均）
    ratio_stats = pd.Series({
        "original_mean_pronoun_ratio": df["orig_pronoun_ratio"].mean(),
        "rewritten_mean_pronoun_ratio": df["rew_pronoun_ratio"].mean(),
    })

    # 代词整体比例（总 pronoun / 总 token）
    orig_total_pron = df["orig_pronoun_cnt"].sum()
    rew_total_pron = df["rew_pronoun_cnt"].sum()
    orig_total_tok = df["orig_len"].sum()
    rew_total_tok = df["rew_len"].sum()

    overall_ratio = pd.Series({
        "original_overall_pronoun_ratio": orig_total_pron / orig_total_tok if orig_total_tok else np.nan,
        "rewritten_overall_pronoun_ratio": rew_total_pron / rew_total_tok if rew_total_tok else np.nan,
    })

    print("=" * 80)
    print(f"Domain: {domain}")
    print("\n=== 长度统计（token）===")
    display(length_stats)

    print("\n=== 逐样本代词比例的平均值 ===")
    display(ratio_stats.to_frame(name="value"))

    print("\n=== 整体代词比例 (总 pronoun / 总 token) ===")
    display(overall_ratio.to_frame(name="value"))

    print("\n=== 原始查询代词计数 ===")
    print(orig_pron_counter)

    print("\n=== 改写查询代词计数 ===")
    print(rew_pron_counter)

    print("\n示例数据预览：")
    preview_cols = [f"{orig_field}_orig", f"{rew_field}_rew", "orig_len", "rew_len", "orig_pronoun_ratio", "rew_pronoun_ratio"]
    display(df[preview_cols].head())


# 依次对每个 domain 进行统计
for d in domains:
    analyze_domain(d)


Domain: clapnq

=== 长度统计（token）===


,mean_len,median_len
original,10.920398,10.0
rewritten,10.402985,10.0



=== 逐样本代词比例的平均值 ===


,value
original_mean_pronoun_ratio,0.015508
rewritten_mean_pronoun_ratio,0.009904



=== 整体代词比例 (总 pronoun / 总 token) ===


...

,value
original_mean_pronoun_ratio,0.011508
rewritten_mean_pronoun_ratio,0.009615



=== 整体代词比例 (总 pronoun / 总 token) ===


,value
original_overall_pronoun_ratio,0.012373
rewritten_overall_pronoun_ratio,0.010480



=== 原始查询代词计数 ===
Counter({'that': 14, 'it': 7, 'this': 3, 'they': 2, 'these': 2})

=== 改写查询代词计数 ===
Counter({'it': 8, 'that': 7, 'they': 3, 'those': 2, 'this': 2, 'these': 2})

示例数据预览：


,text_orig,text_rew,orig_len,rew_len,orig_pronoun_ratio,rew_pronoun_ratio
0,"|user|: ""What are the sheltered rooms designat...",|user|: What are sheltered rooms designated fo...,9,8,0.0,0.0
1,|user|: What items should I keep in the safe r...,|user|: What items should I keep in a designat...,10,11,0.0,0.0
2,|user|: Is the same three-day supply recommend...,|user|: Are sheltered room designations and re...,10,12,0.0,0.0
3,|user|: Which state experiences more wildfires?,|user|: Which state has more wildfires?,6,6,0.0,0.0
4,|user|: What causes wildfires?,|user|: What causes wildfires?,4,4,0.0,0.0
